# F1 Pit Stop Ensemble — Mac M1 版本

In [11]:
# 只需執行一次，裝好後可以註解掉
!pip install ydf pytabkit category_encoders

In [23]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import re
import math
import random
import contextlib
import io
from copy import deepcopy
from functools import partial
from itertools import combinations
from pprint import pprint
from typing import Any, Literal, NamedTuple, Optional, Dict
import lightgbm as lgb
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder,
    label_binarize, OrdinalEncoder, QuantileTransformer, TargetEncoder,
    RobustScaler, FunctionTransformer, KBinsDiscretizer
)
from category_encoders import CatBoostEncoder, MEstimateEncoder
from sklearn.ensemble import (
    RandomForestClassifier, VotingClassifier, HistGradientBoostingClassifier,
    GradientBoostingClassifier
)
from sklearn.linear_model import (
    RidgeClassifier, LogisticRegression, LinearRegression, Ridge
)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold, cross_val_score
)
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, mean_squared_error,
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, r2_score, balanced_accuracy_score, average_precision_score
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.inspection import permutation_importance
from scipy.stats import norm, skew
from scipy.special import expit, logit
from scipy.optimize import minimize
from colorama import Fore, Style, init
from tqdm import tqdm
from IPython.display import display_html, clear_output

from xgboost import DMatrix, XGBClassifier, XGBRegressor
import xgboost as xgb
from lightgbm import log_evaluation, early_stopping, LGBMClassifier, LGBMRegressor, Dataset
import lightgbm
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

import torch

# pytabkit
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier

# ydf
import ydf

# Keras / TF
import keras
from keras.models import Sequential
from keras import layers
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.layers import (
    Lambda, Input, Embedding, Flatten, Dense, Concatenate, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, AdamW

print("✅ 所有套件載入完成")
print(f"PyTorch: {torch.__version__}")
print(f"MPS 可用: {torch.backends.mps.is_available()}")
print(f"TensorFlow: {tf.__version__}")

✅ 所有套件載入完成
PyTorch: 2.12.0
MPS 可用: True
TensorFlow: 2.21.0


## Configuration

In [24]:
class Config:
    # ===== 請修改成你的資料路徑 =====
    DATA_DIR = '/Users/test/Documents/清華大學/三下/機器學習概論/hw5/data'
    # ================================

    target = 'PitNextLap'
    train = pd.read_csv(f'{DATA_DIR}/train.csv', index_col='id')
    test  = pd.read_csv(f'{DATA_DIR}/test.csv',  index_col='id')
    submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
    orig  = pd.read_csv(f'{DATA_DIR}/f1_strategy_dataset_v4.csv')[train.columns]

    # Mac M1：不用 CUDA，用 MPS 或 CPU
    device     = 'mps' if torch.backends.mps.is_available() else 'cpu'
    xgb_device = 'cpu'   # XGBoost 在 Mac 只支援 cpu
    cat_device = 'CPU'   # CatBoost 大寫

    state    = 42
    n_splits = 5
    early_stop = 100
    metric   = 'roc_auc'
    task_type = 'binary'
    task_is_regression = task_type == 'regression'

    if task_is_regression:
        n_classes = 1
        folds = KFold(n_splits=n_splits, shuffle=True, random_state=state)
    else:
        n_classes = 2 if task_type == 'binary' else train[target].nunique()
        labels = train[target].unique()
        folds  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=state)

print("✅ Config 載入完成")
print(f"Train: {Config.train.shape}, Test: {Config.test.shape}")
print(f"Device: {Config.device}")

✅ Config 載入完成
Train: (439140, 15), Test: (188165, 14)
Device: mps


## EDA（可選，跳過不影響訓練）

In [14]:
class EDA(Config):
    def __init__(self):
        super().__init__()
        self.cat_features = self.train.drop(self.target, axis=1).select_dtypes(include=['object','bool']).columns.tolist()
        self.num_features = self.train.drop(self.target, axis=1).select_dtypes(exclude=['object','bool']).columns.tolist()
        self.data_info()
        self.heatmap()
        self.dist_plots()
        self.cat_feature_plots()
        self.target_pie()

    def data_info(self):
        for data, label in zip([self.train, self.test], ['Train', 'Test']):
            print(Style.BRIGHT+Fore.GREEN+f'\n{label} head\n')
            display(data.head())
            print(Style.BRIGHT+Fore.GREEN+f'\n{label} describe\n')
            display(data.describe().T)
            print(Style.BRIGHT+Fore.GREEN+f'\n{label} missing values\n'+Style.RESET_ALL)
            display(data.isna().sum())
        return self

    def heatmap(self):
        plt.figure(figsize=(10,10))
        corr = self.train[self.num_features+[self.target]].corr(method='spearman')
        sns.heatmap(corr, fmt='0.2f', cmap='Greens', square=True, annot=True, linewidths=1, cbar=False)
        plt.show()

    def dist_plots(self):
        df = pd.concat([self.train[self.num_features].assign(Source='Train'),
                        self.test[self.num_features].assign(Source='Test')], ignore_index=True)
        fig, axes = plt.subplots(len(self.num_features), 2, figsize=(18, len(self.num_features)*6))
        for i, col in enumerate(self.num_features):
            sns.kdeplot(data=df[[col,'Source']], x=col, hue='Source',
                        palette=['#3cb371','#ef5350'], ax=axes[i,0], linewidth=2)
            sns.boxplot(data=df, y=col, x='Source', ax=axes[i,1],
                        palette=['#3cb371','#ef5350'])
        plt.tight_layout()
        plt.show()

    def cat_feature_plots(self):
        if not self.cat_features:
            return
        fig, axes = plt.subplots(len(self.cat_features), 2, figsize=(18, len(self.cat_features)*6))
        if len(self.cat_features) == 1:
            axes = np.array([axes])
        for i, col in enumerate(self.cat_features):
            sns.barplot(data=self.train[col].value_counts().nlargest(10).reset_index(),
                        x=col, y='count', ax=axes[i,0], color='#3cb371')
            sns.barplot(data=self.test[col].value_counts().nlargest(10).reset_index(),
                        x=col, y='count', ax=axes[i,1], color='#ef5350')
        plt.tight_layout()
        plt.show()

    def target_pie(self):
        counts = self.train[self.target].value_counts(normalize=True)
        plt.figure(figsize=(6,6))
        plt.pie(counts.values, labels=counts.index, autopct='%1.2f%%',
                colors=['#5c9ded','#3cb371'])
        plt.show()

# 需要 EDA 時取消註解
# eda = EDA()

## Preprocessing

In [25]:
class Preprocessing:
    def __init__(self, target):
        self.target  = target
        self._fitted = False

    def fit_transform(self, X_train, y_train, orig=None):
        self.fit(X_train, y_train, orig=orig)
        return self.transform(X_train)

    def fit(self, X_train, y_train, orig=None):
        self.X_train_ = X_train.copy()
        self.y_train_ = y_train.copy()
        self.num_features_ = self.X_train_.select_dtypes(exclude=['object','bool','category']).columns.tolist()
        self.cat_features_ = self.X_train_.select_dtypes(include=['object','bool','category']).columns.tolist()
        self._orig_stats = {}
        if orig is not None:
            self.orig_ = orig.copy()
            self._fit_orig_target_stats(self.cat_features_ + self.num_features_)
        self._fit_feature_engineering(self.X_train_)
        X_fe_train = self._apply_feature_engineering(self.X_train_)
        self._fit_frequency_encoding(X_fe_train)
        X_ready = self._apply_frequency_encoding(X_fe_train)
        X_ready = self._finalize_types(X_ready, fit=True)
        self._fitted = True
        return self

    def transform(self, X):
        if not self._fitted:
            raise RuntimeError("Call fit() first.")
        X = X.copy()
        X = self._apply_orig_target_stats(X)
        X = self._apply_feature_engineering(X)
        X = self._apply_frequency_encoding(X)
        X = self._finalize_types(X, fit=False)
        for c in self.cat_features:
            X[c] = X[c].astype(object).fillna('NaN').astype('category')
        return X, self.cat_features, self.num_features

    def _fit_orig_target_stats(self, cols):
        self._orig_global_mean = float(self.orig_[self.target].mean())
        for c in cols:
            if c in self.orig_.columns:
                self._orig_stats[c] = self.orig_.groupby(c, observed=False)[self.target].mean()
            else:
                self._orig_stats[c] = pd.Series(dtype=float)

    def _apply_orig_target_stats(self, df):
        df = df.copy()
        for c, ser in self._orig_stats.items():
            col = f"{c}_org_mean"
            if c in df.columns and not ser.empty:
                df[col] = df[c].map(ser).astype(float).fillna(self._orig_global_mean)
            else:
                df[col] = self._orig_global_mean
        return df

    def _fit_feature_engineering(self, X_train):
        X_train = X_train.copy()
        self._highcard_num = [c for c in self.num_features_ if X_train[c].nunique(dropna=False) > 20]
        self.category_map = {}
        bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
        for col, bins_list in bin_config.items():
            for n_bins in bins_list:
                for strategy in ['quantile']:
                    bin_name = f"{col}_{n_bins}_{strategy}_bin"
                    kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy=strategy, subsample=None)
                    kb.fit_transform(X_train[[col]]).ravel().astype('int32')
                    self.category_map[bin_name] = kb
        X_train['TyreLife_LapNumber_ratio'] = (X_train['TyreLife'] / X_train['LapNumber'].clip(lower=1)).astype('float32')
        X_train['LapNumber_RaceProgress_ratio'] = (X_train['LapNumber'] / (X_train['RaceProgress'] + 1e-6)).astype('float32')
        self.num_to_cat = []
        for col in self.num_features_ + ['TyreLife_LapNumber_ratio', 'LapNumber_RaceProgress_ratio']:
            cat_name = f"{col}_cat"
            codes, uniques = np.floor(X_train[col]).factorize()
            self.category_map[col] = uniques
            X_train[cat_name] = codes.astype(str)
            self.num_to_cat.append(cat_name)
        for col in self.cat_features_ + self.num_to_cat:
            count_name = f"{col}_count"
            count_map = X_train[col].value_counts()
            X_train[count_name] = X_train[col].map(count_map).fillna(0).astype('int32')
            self.category_map[count_name] = count_map

    def _apply_feature_engineering(self, df):
        df = df.copy()
        df['Race_Year_comb']    = df['Race'].astype(str) + '_' + df['Year'].astype(str)
        df['Race_Compound_comb']= df['Race'].astype(str) + '_' + df['Compound'].astype(str)
        df['TyreLife_LapNumber_ratio'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
        df['LapNumber_RaceProgress_ratio'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
        df['LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
        df['LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
        df['LapTime (s)_Cumulative_Degradation_abs_ratio'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')
        bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
        for col, bins_list in bin_config.items():
            for n_bins in bins_list:
                for strategy in ['quantile']:
                    bin_name = f"{col}_{n_bins}_{strategy}_bin"
                    kb = self.category_map[bin_name]
                    df[bin_name] = kb.transform(df[[col]]).ravel().astype('int32')
                    df[bin_name] = df[bin_name].astype('category')
        for col in self.num_features_ + ['TyreLife_LapNumber_ratio', 'LapNumber_RaceProgress_ratio']:
            cat_name = f"{col}_cat"
            uniques  = self.category_map.get(col, [])
            code_map = {cat: i for i, cat in enumerate(uniques)}
            mapped   = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
            df[cat_name] = mapped.astype(str)
        for col in self.cat_features_ + self.num_to_cat:
            count_name = f"{col}_count"
            count_map  = self.category_map[count_name]
            df[count_name] = df[col].map(count_map).fillna(0).astype('int32')
        drop = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
        if drop:
            df.drop(drop, axis=1, inplace=True)
        return df

    def _fit_frequency_encoding(self, X):
        all_cats = list(dict.fromkeys(self.cat_features_ + self.num_to_cat))
        self._fe_cols = [c for c in all_cats if c in X.columns]
        self._freq_encodings = {}
        for c in self._fe_cols:
            self._freq_encodings[c] = X[c].value_counts(normalize=True).to_dict()

    def _apply_frequency_encoding(self, X):
        X = X.copy()
        for c in getattr(self, '_fe_cols', []):
            mapping = self._freq_encodings.get(c, {})
            if c in X.columns:
                X[f"{c}_fe"] = X[c].map(mapping).astype(float).fillna(0.0)
            else:
                X[f"{c}_fe"] = 0.0
        return X

    def _finalize_types(self, X, fit=False):
        self.num_features = X.select_dtypes(exclude=['object','bool','category']).columns.tolist()
        self.cat_features = X.select_dtypes(include=['object','bool','category']).columns.tolist()
        return X

In [26]:
y      = Config.train[Config.target]
X      = Config.train.drop(Config.target, axis=1)
orig_y = Config.orig[Config.target]
orig_X = Config.orig.drop(Config.target, axis=1)

p = Preprocessing(Config.target)
X,      cat_features, num_features = p.fit_transform(X, y, Config.orig)
test,   _,            _            = p.transform(Config.test)
orig_X, _,            _            = p.transform(orig_X)

print(f"✅ Preprocessing 完成")
print(f"X: {X.shape}, test: {test.shape}, orig_X: {orig_X.shape}")
print(f"cat_features: {len(cat_features)}, num_features: {len(num_features)}")

✅ Preprocessing 完成
X: (439140, 82), test: (188165, 82), orig_X: (101371, 82)
cat_features: 20, num_features: 62


## Models

In [27]:
# ── YDF wrapper ──────────────────────────────────────────────────────────────
class YDFModel(BaseEstimator, Config):
    def __init__(self, learner_type='RF', params=None):
        self.params = {} if params is None else params.copy()
        if learner_type == 'RF':
            self.learner_class = ydf.RandomForestLearner
        elif learner_type == 'GBT':
            self.learner_class = ydf.GradientBoostedTreesLearner
        else:
            raise ValueError(f'Unknown learner_type: {learner_type}')
        self.model = None

    def fit(self, X, y, X_val=None, y_val=None, sample_weight=None):
        if not self.task_is_regression:
            y = y.astype(int)
            if y_val is not None:
                y_val = y_val.astype(int)
        self.n_classes = 1 if self.task_is_regression else (2 if len(np.unique(y)) == 2 else len(np.unique(y)))
        target = y.name
        params = self.params.copy()
        params['label'] = target
        params['task']  = ydf.Task.REGRESSION if self.task_is_regression else ydf.Task.CLASSIFICATION
        train_df = X.copy(); train_df[target] = y.values
        valid_df = None
        if X_val is not None and y_val is not None:
            valid_df = X_val.copy(); valid_df[target] = y_val.values
        supports_valid = self.learner_class is ydf.GradientBoostedTreesLearner
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            learner = self.learner_class(**params)
            if supports_valid and valid_df is not None:
                try:
                    self.model = learner.train(train_df, valid=valid_df, sample_weights=sample_weight)
                except TypeError:
                    self.model = learner.train(train_df, valid=valid_df)
            else:
                try:
                    self.model = learner.train(train_df, sample_weights=sample_weight)
                except TypeError:
                    self.model = learner.train(train_df)
        return self

    def _ensure_proba_shape(self, raw_pred):
        proba = np.asarray(raw_pred)
        if not self.task_is_regression and self.n_classes == 2:
            if proba.ndim == 1:
                return np.vstack([1-proba, proba]).T
            elif proba.ndim == 2 and proba.shape[1] == 1:
                return np.hstack([1-proba, proba])
        if proba.ndim == 1:
            return proba.reshape(-1,1)
        return proba

    def predict(self, X):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            raw = self.model.predict(X)
        if self.task_is_regression:
            return np.asarray(raw)
        return self._ensure_proba_shape(raw)

    def predict_proba(self, X):
        if self.task_is_regression:
            raise AttributeError('predict_proba not available for regression')
        return self.predict(X)

In [28]:
# ── Keras NN wrapper ──────────────────────────────────────────────────────────
class KerasTabularModel(BaseEstimator, Config):
    def __init__(self, embedding_dim_func=None, hidden_units=[256,128,64],
                 dropout=0.3, learning_rate=1e-3, epochs=20, batch_size=64,
                 early_stopping_patience=3, reduce_lr_patience=1,
                 cat_features=[], num_features=[]):
        self.embedding_dim_func = embedding_dim_func or (lambda c: int(np.ceil(np.sqrt(c))))
        self.hidden_units = hidden_units
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.cat_features = cat_features
        self.num_features = num_features
        self.early_stopping_patience = early_stopping_patience
        self.reduce_lr_patience = reduce_lr_patience

    def _build_model(self):
        cat_input = Input(shape=(len(self.cat_features),), name='cat_input')
        num_input = Input(shape=(len(self.num_features),), name='num_input')
        embs = []
        for j, card in enumerate(self.cat_cardinalities):
            emb_dim = int(np.ceil(np.sqrt(card)))
            xj = layers.Embedding(input_dim=card, output_dim=emb_dim, name=f'emb_{j}')(cat_input[:,j])
            embs.append(layers.Flatten()(xj))
        x = layers.Concatenate(axis=-1, name='all_tokens')(embs + [num_input])
        for idx, units in enumerate(self.hidden_units):
            x = layers.Dense(units, activation='relu',
                             kernel_regularizer=tf.keras.regularizers.l2(1e-5),
                             name=f'dense_{idx}')(x)
            x = layers.Dropout(self.dropout, name=f'drop_{idx}')(x)
            x = layers.BatchNormalization(name=f'bn_{idx}')(x)
        if self.task_type == 'regression':
            output = layers.Dense(1, activation='linear', name='output')(x)
            loss, metrics = 'mse', ['mse']
        elif self.task_type == 'binary':
            output = layers.Dense(1, activation='sigmoid', name='output')(x)
            loss, metrics = 'binary_crossentropy', ['accuracy']
        else:
            output = layers.Dense(self.n_classes, activation='softmax', name='output')(x)
            loss, metrics = 'sparse_categorical_crossentropy', ['accuracy']
        self.model = Model(inputs=[cat_input, num_input], outputs=output)
        self.model.compile(optimizer=AdamW(learning_rate=self.learning_rate, weight_decay=1e-5),
                           loss=loss, metrics=metrics)

    def _process_X(self, X, training=False):
        X_cat = X[self.cat_features]
        X_num = X[self.num_features] if self.num_features else np.zeros((len(X), 0))
        if training:
            self._qt = QuantileTransformer(
                n_quantiles=max(min(len(X)//30, 1000), 10),
                output_distribution='normal', subsample=10**9)
            X_num = self._qt.fit_transform(X_num)
            self._oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_cat = self._oe.fit_transform(X_cat)
        else:
            X_num = self._qt.transform(X_num)
            X_cat = self._oe.transform(X_cat)
        return [X_cat, X_num]

    def fit(self, X, y, eval_set=None, verbose=0):
        self.cat_cardinalities = [X[col].nunique() for col in self.cat_features]
        X_proc = self._process_X(X, training=True)
        self._build_model()
        val_data = None
        if eval_set is not None:
            X_val, y_val = eval_set[0]
            val_data = (self._process_X(X_val), y_val)
        return self.model.fit(X_proc, y, epochs=self.epochs, batch_size=self.batch_size,
                              validation_data=val_data, verbose=verbose,
                              callbacks=[
                                  keras.callbacks.ReduceLROnPlateau(patience=self.reduce_lr_patience),
                                  keras.callbacks.EarlyStopping(patience=self.early_stopping_patience,
                                                                restore_best_weights=True)])

    def predict(self, X):
        preds = self.model.predict(self._process_X(X), verbose=0)
        if self.task_type == 'regression': return preds.squeeze()
        elif self.task_type == 'binary':   return (preds.squeeze() > 0.5).astype(int)
        else:                              return preds.argmax(axis=1)

    def predict_proba(self, X):
        preds = self.model.predict(self._process_X(X), verbose=0)
        if self.task_type == 'binary':
            preds = preds.squeeze()
            return np.vstack([1-preds, preds]).T
        return preds

In [29]:
# ── Models dict（Mac M1 版：全部 CPU，max_bin 縮小）──────────────────────────
models = {
    'XGB_16': XGBClassifier(**{
        'n_estimators': 10000,
        'learning_rate': 0.01,
        'max_depth': 6,
        'random_state': Config.state,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'enable_categorical': True,
        'device': Config.xgb_device,   # 'cpu'
        'max_bin': 255,                # 原本 5000 → 改 255
    }),
    'XGB2_21': XGBClassifier(**{
        'n_estimators': 10000,
        'learning_rate': 0.01,
        'random_state': Config.state,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'enable_categorical': True,
        'device': Config.xgb_device,
        'max_bin': 255,
        'reg_lambda': 8.162374349037115,
        'reg_alpha': 8.354463958574286,
        'colsample_bytree': 0.1450999139156032,
        'subsample': 0.8570122278990485,
        'max_depth': 10,
        'min_child_weight': 2,
    }),
    'XGB3_2': XGBClassifier(**{
        'n_estimators': 10000,
        'learning_rate': 0.01,
        'random_state': Config.state,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'enable_categorical': True,
        'device': Config.xgb_device,
        'max_bin': 255,
        'reg_lambda': 7.336548282259911,
        'reg_alpha': 4.717742768334144,
        'colsample_bytree': 0.2467816540614318,
        'subsample': 0.8468669571395848,
        'max_depth': 11,
        'min_child_weight': 6,
    }),
    'XGB4': XGBClassifier(**{
        'n_estimators': 10000,
        'learning_rate': 0.01,
        'random_state': Config.state,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'enable_categorical': True,
        'device': Config.xgb_device,
        'max_bin': 255,
        'reg_lambda': 0.25356710846329644,
        'reg_alpha': 3.9431008774671916,
        'colsample_bytree': 0.17774825158281427,
        'subsample': 0.6948146928263638,
        'max_depth': 14,
        'min_child_weight': 9,
        'verbosity': 2,          # 0=silent, 1=warning, 2=info, 3=debug
        'verbose': True,
    }),
    'Realmlp_2': RealMLP_TD_Classifier(**{
        'val_metric_name': '1-balanced_accuracy',
        'device': Config.device,       # 'mps' 或 'cpu'
        'random_state': 42,
        'verbosity': 0,
        'n_epochs': 100,
        'batch_size': 256,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 10,
        'early_stopping_multiplicative_patience': 1,
        'act': 'mish',
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': 'rectangular',
        'hidden_width': 384,
        'lr': 0.04,
        'ls_eps': 0.011498317194338772,
        'ls_eps_sched': 'coslog4',
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 4,
        'p_drop': 0.07301419697186451,
        'p_drop_sched': 'flat_cos',
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }),
    'Realmlp2_3': RealMLP_TD_Classifier(**{
        'random_state': 42,
        'verbosity': 0,
        'val_metric_name': '1-balanced_accuracy',
        'device': Config.device,
        'n_ens': 8,
        'n_epochs': 3,
        'batch_size': 256,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 10,
        'early_stopping_multiplicative_patience': 1,
        'lr': 0.075, 'wd': 0.0236, 'sq_mom': 0.988,
        'lr_sched': 'flat_anneal', 'first_layer_lr_factor': 0.25,
        'add_front_scale': False, 'bias_init_mode': 'neg-uniform-dynamic-2',
        'embedding_size': 6, 'max_one_hot_cat_size': 18, 'act': 'silu',
        'p_drop': 0.05, 'p_drop_sched': 'expm4t',
        'plr_hidden_1': 16, 'plr_hidden_2': 8, 'plr_act_name': 'gelu',
        'plr_lr_factor': 0.1151, 'plr_sigma': 2.33,
        'ls_eps': 0.01, 'ls_eps_sched': 'sqrt_cos',
    }),
    'Realmlp3_8': RealMLP_TD_Classifier(**{
        'random_state': 42, 'verbosity': 0,
        'val_metric_name': '1-auc_ovr',
        'device': Config.device,
        'n_ens': 8, 'n_epochs': 4, 'batch_size': 256,
        'use_early_stopping': False,
        'early_stopping_additive_patience': 10,
        'early_stopping_multiplicative_patience': 1,
        'lr': 0.07, 'wd': 0.018, 'sq_mom': 0.98,
        'lr_sched': 'cos_anneal', 'first_layer_lr_factor': 0.25,
        'embedding_size': 6, 'max_one_hot_cat_size': 18,
        'hidden_sizes': [512,256,128], 'act': 'silu',
        'p_drop': 0.05, 'p_drop_sched': 'expm4t',
        'plr_hidden_1': 16, 'plr_hidden_2': 8, 'plr_act_name': 'gelu',
        'plr_lr_factor': 0.1151, 'plr_sigma': 2.33,
        'ls_eps': 0.01, 'ls_eps_sched': 'sqrt_cos',
        'add_front_scale': False, 'bias_init_mode': 'neg-uniform-dynamic-2',
        'tfms': ['one_hot','median_center','robust_scale','smooth_clip','embedding','l2_normalize'],
    }),
    'Realmlp4_7': RealMLP_TD_Classifier(**{
        'random_state': 42, 'verbosity': 2,
        'val_metric_name': '1-auc_ovr',
        'device': Config.device,
        'n_ens': 24, 'n_epochs': 6, 'batch_size': 256,
        'use_early_stopping': False,
        'early_stopping_additive_patience': 10,
        'early_stopping_multiplicative_patience': 1,
        'lr': 0.01, 'wd': 0.016, 'sq_mom': 0.99,
        'lr_sched': 'lin_cos_log_15', 'first_layer_lr_factor': 0.25,
        'embedding_size': 6, 'max_one_hot_cat_size': 18,
        'hidden_sizes': [512,256,128], 'act': 'silu',
        'p_drop': 0.05, 'p_drop_sched': 'expm4t',
        'plr_hidden_1': 16, 'plr_hidden_2': 8, 'plr_act_name': 'gelu',
        'plr_lr_factor': 0.1151, 'plr_sigma': 2.33,
        'ls_eps': 0.01, 'ls_eps_sched': 'sqrt_cos',
        'add_front_scale': False, 'bias_init_mode': 'neg-uniform-dynamic-2',
        'tfms': ['one_hot','median_center','robust_scale','smooth_clip','embedding','l2_normalize'],
    }),
    'LGBM_3': LGBMClassifier(**{
        'random_state': Config.state,
        'early_stopping_round': Config.early_stop,
        'verbose': -1,
        'n_estimators': 20000,
        'metric': 'auc',
        'objective': 'binary',
        'learning_rate': 0.01,
        'max_depth': 6,
        'max_bin': 255,
    }),
    'LGBM2_12': LGBMClassifier(**{
        'random_state': Config.state,
        'early_stopping_round': Config.early_stop,
        'verbose': -1,
        'n_estimators': 20000,
        'metric': 'auc',
        'objective': 'binary',
        'learning_rate': 0.01,
        'max_bin': 255,
        'max_depth': 11,
        'min_child_samples': 94,
        'subsample': 0.6522179123688497,
        'colsample_bytree': 0.32842983929259567,
        'num_leaves': 140,
        'reg_alpha': 1.0377177748954165,
        'reg_lambda': 0.1613822177468191,
    }),
    'LGBM3_2': LGBMClassifier(**{
        'random_state': Config.state,
        'early_stopping_round': Config.early_stop,
        'verbose': -1,
        'n_estimators': 20000,
        'metric': 'auc',
        'objective': 'binary',
        'learning_rate': 0.01,
        'max_bin': 255,
        'feature_pre_filter': False,
        'max_depth': 12,
        'min_child_samples': 33,
        'subsample': 0.26685144787781945,
        'colsample_bytree': 0.23457900771427878,
        'num_leaves': 381,
        'reg_alpha': 5.499071258569893,
        'reg_lambda': 0.02547086863080793,
    }),
    'LGBM4': LGBMClassifier(**{
        'random_state': Config.state,
        'early_stopping_round': Config.early_stop,
        'verbose': -1,
        'n_estimators': 20000,
        'metric': 'auc',
        'objective': 'binary',
        'learning_rate': 0.01,
        'max_bin': 255,
        'max_depth': 14,
        'min_child_samples': 136,
        'subsample': 0.49213571528068917,
        'colsample_bytree': 0.2003244975377485,
        'num_leaves': 431,
        'reg_alpha': 0.10964422705802229,
        'reg_lambda': 6.321652986460238,
    }),
    'LR_6': LogisticRegression(
        penalty='l2', solver='lbfgs', C=1.0,
        random_state=Config.state, max_iter=1000,
        class_weight='balanced', n_jobs=-1,
    ),
    'CAT_2': CatBoostClassifier(**{
        'verbose': 0,
        'loss_function': 'Logloss',
        'random_state': Config.state,
        'early_stopping_rounds': Config.early_stop,
        'eval_metric': 'AUC',
        'n_estimators': 20000,
        'learning_rate': 0.01,
        'task_type': Config.cat_device,   # 'CPU'
        'depth': 8,
    }),
    'CAT2_11': CatBoostClassifier(**{
        'verbose': 0,
        'loss_function': 'Logloss',
        'random_state': Config.state,
        'early_stopping_rounds': Config.early_stop,
        'eval_metric': 'AUC',
        'n_estimators': 20000,
        'learning_rate': 0.05,
        'task_type': Config.cat_device,   # 修正：原本寫成字串 'Config.cat_device'
        'depth': 10,
        'min_data_in_leaf': 54,
        'l2_leaf_reg': 8.04214109088981,
        'bagging_temperature': 0.12529500670009353,
        'random_strength': 8.965600403402266,
    }),
    'CAT4_2': CatBoostClassifier(**{
        'verbose': 0,
        'loss_function': 'Logloss',
        'random_state': Config.state,
        'early_stopping_rounds': Config.early_stop,
        'eval_metric': 'AUC',
        'n_estimators': 20000,
        'learning_rate': 0.01,
        'task_type': Config.cat_device,
        'depth': 9,
        'min_data_in_leaf': 26,
        'l2_leaf_reg': 6.457023621519966,
        'bagging_temperature': 0.5995185332846042,
        'random_strength': 0.27483928483377495,
    }),
    'NN_3': KerasTabularModel(
        hidden_units=[256,128],
        dropout=0.1,
        epochs=30,
        learning_rate=1e-3,
        batch_size=1000,
    ),
    'HGB_3': HistGradientBoostingClassifier(**{
        'max_iter': 20000,
        'random_state': Config.state,
        'early_stopping': True,
        'categorical_features': 'from_dtype',
        'learning_rate': 0.01,
        'loss': 'log_loss',
        'scoring': 'roc_auc',
        'tol': 1e-7,
        'l2_regularization': 9.206887135749392,
        'max_depth': 10,
        'max_leaf_nodes': 190,
        'min_samples_leaf': 65,
    }),
    'YDF_GBT_3': YDFModel(learner_type='GBT', params={
        'num_trees': 600,
        'random_seed': Config.state,
        'growing_strategy': 'BEST_FIRST_GLOBAL',
        'shrinkage': 0.1,
        'early_stopping_num_trees_look_ahead': 50,
        'categorical_algorithm': 'CART',
        'max_depth': 3,
        'min_examples': 10,
    }),
    'TabMD_4': TabM_D_Classifier(**{
        'batch_size': 'auto',
        'patience': 16,
        'allow_amp': True,
        'arch_type': 'tabm-mini',
        'tabm_k': 8,
        'gradient_clipping_norm': 1.0,
        'share_training_batches': False,
        'lr': 1e-3,
        'weight_decay': 1e-3,
        'n_blocks': 3,
        'd_block': 512,
        'dropout': 0.0,
        'num_emb_type': 'pwl',
        'd_embedding': 32,
        'num_emb_n_bins': 119,
    }),
}

print(f"✅ Models 定義完成，共 {len(models)} 個模型")
print(list(models.keys()))

✅ Models 定義完成，共 20 個模型
['XGB_16', 'XGB2_21', 'XGB3_2', 'XGB4', 'Realmlp_2', 'Realmlp2_3', 'Realmlp3_8', 'Realmlp4_7', 'LGBM_3', 'LGBM2_12', 'LGBM3_2', 'LGBM4', 'LR_6', 'CAT_2', 'CAT2_11', 'CAT4_2', 'NN_3', 'HGB_3', 'YDF_GBT_3', 'TabMD_4']


## Training

In [30]:
class FeatureEncoder:
    def __init__(self, num_features, cat_features):
        self.num_features = num_features
        self.cat_features = cat_features
        self.ohe   = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.scaler= StandardScaler()
        self.ohe_cols = None

    def fit(self, X):
        self.ohe.fit(X[self.cat_features])
        self.ohe_cols = self.ohe.get_feature_names_out(self.cat_features)
        self.scaler.fit(X[self.num_features])

    def transform_fold(self, X_train, X_val, X_test):
        def transform(X):
            X = X.copy()
            X[self.num_features] = self.scaler.transform(X[self.num_features])
            X_ohe = pd.DataFrame(self.ohe.transform(X[self.cat_features]),
                                 columns=self.ohe_cols, index=X.index)
            X = pd.concat([X.drop(columns=self.cat_features).reset_index(drop=True),
                           X_ohe.reset_index(drop=True)], axis=1)
            return X
        return transform(X_train), transform(X_val), transform(X_test)

In [ ]:
class Trainer(Config):
    def __init__(self, X, y, test, models, num_features, cat_features, training=True):
        self.X = X
        self.test = test
        self.y = y
        self.models = models
        self.training = training
        self.scores    = pd.DataFrame(columns=['Score'], dtype=float)
        self.OOF_preds = pd.DataFrame(dtype=float)
        self.TEST_preds= pd.DataFrame(dtype=float)
        self.num_features = num_features
        self.cat_features = cat_features

    def ScoreMetric(self, y_true, y_pred):
        if self.task_type == 'multiclass':
            y_label = np.argmax(y_pred, axis=1)
        elif self.task_type == 'binary':
            y_label = (y_pred >= 0.5).astype(int)
        else:
            y_label = y_pred
        if self.metric == 'roc_auc':
            return roc_auc_score(y_true, y_pred, multi_class='ovr') if self.task_type == 'multiclass' else roc_auc_score(y_true, y_pred)
        if self.metric == 'accuracy':
            return accuracy_score(y_true, y_label)
        if self.metric == 'balanced_accuracy':
            return balanced_accuracy_score(y_true, y_label)
        if self.metric == 'r2':
            return r2_score(y_true, y_label)

    def train(self, model, X, y, test, model_name):
        if self.task_type == 'multiclass':
            oof_pred  = np.zeros((X.shape[0], self.n_classes), dtype=float)
            test_pred = np.zeros((test.shape[0], self.n_classes), dtype=float)
        else:
            oof_pred  = np.zeros(X.shape[0], dtype=float)
            test_pred = np.zeros(test.shape[0], dtype=float)

        print('='*20)
        print(model_name)
        params = model.get_params()

        for n_fold, (train_id, valid_id) in enumerate(self.folds.split(X, y)):
            X_train = X.iloc[train_id].copy()
            y_train = y.iloc[train_id]
            X_val   = X.iloc[valid_id].copy()
            y_val   = y.iloc[valid_id]
            X_test  = test.copy()

            if model_name != 'Ensemble':
                X_train = pd.concat([X_train, orig_X], axis=0).reset_index(drop=True)
                y_train = pd.concat([y_train, orig_y], axis=0).reset_index(drop=True)
                X_train[self.cat_features] = X_train[self.cat_features].astype('category')

                # ✅ 修正：一次處理所有 cat 欄位，避免記憶體爆炸
                te = TargetEncoder(random_state=42, shuffle=True, cv=self.n_splits,
                                   smooth='auto', target_type='binary')
                te_cols = self.cat_features
                te_train = te.fit_transform(X_train[te_cols], y_train).astype('float32')
                te_val   = te.transform(X_val[te_cols]).astype('float32')
                te_test  = te.transform(X_test[te_cols]).astype('float32')
                for i, col in enumerate(te_cols):
                    X_train[f'{col}_te'] = te_train[:, i]
                    X_val[f'{col}_te']   = te_val[:, i]
                    X_test[f'{col}_te']  = te_test[:, i]
                del te_train, te_val, te_test
                X_train = X_train.drop(self.cat_features, axis=1)
                X_val   = X_val.drop(self.cat_features, axis=1)
                X_test  = X_test.drop(self.cat_features, axis=1)

            num_features = X_train.select_dtypes(exclude=['category']).columns.tolist()
            cat_features = X_train.select_dtypes(include=['category']).columns.tolist()
            print(f'Fold {n_fold+1}')

            if 'LGBM' in model_name:
                lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
                lgb_val   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=cat_features)
                model = lgb.train(
                    params=params, train_set=lgb_train, valid_sets=[lgb_val],
                    num_boost_round=100_000,
                    callbacks=[log_evaluation(0), early_stopping(self.early_stop, verbose=False)])

            elif 'XGB' in model_name:
                dtrain = DMatrix(X_train, label=y_train, enable_categorical=True)
                dval   = DMatrix(X_val,   label=y_val,   enable_categorical=True)
                dtest  = DMatrix(X_test,                 enable_categorical=True)
                model  = xgb.train(
                    params=params, dtrain=dtrain, evals=[(dval,'valid')],
                    num_boost_round=1000, verbose_eval=False,
                    callbacks=[xgb.callback.EarlyStopping(rounds=self.early_stop, maximize=True, save_best=True)])
                X_val  = dval
                X_test = dtest

            elif any(m in model_name for m in ['FTT','TabMD','ResNet','Realmlp']):
                model.fit(X_train, y_train, X_val, y_val, cat_col_names=cat_features)

            elif any(m in model_name for m in ['NN','TabM']):
                model.num_features = num_features
                model.cat_features = cat_features
                model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

            elif 'CAT' in model_name:
                cat_pool  = Pool(X_train, label=y_train, cat_features=cat_features)
                val_pool  = Pool(X_val,   label=y_val,   cat_features=cat_features)
                test_pool = Pool(X_test,                 cat_features=cat_features)
                model.fit(cat_pool, eval_set=val_pool, verbose=False)
                X_val  = val_pool
                X_test = test_pool

            elif any(m in model_name for m in ['HGB','YDF']):
                model.fit(X_train, y_train, X_val=X_val, y_val=y_val)

            else:
                encoder = FeatureEncoder(num_features=num_features, cat_features=cat_features)
                encoder.fit(X_train)
                X_train, X_val, X_test = encoder.transform_fold(X_train, X_val, X_test)
                model.fit(X_train, y_train)

            if 'XGB' in model_name:
            # xgb.train() returns a Booster; .predict() outputs raw probabilities
                if self.task_type == 'regression':
                    y_pred_val = model.predict(X_val)
                    test_pred += model.predict(X_test) / self.n_splits
                elif self.task_type == 'binary':
                    y_pred_val = model.predict(X_val)          # already a probability scalar
                    test_pred += model.predict(X_test) / self.n_splits
                elif self.task_type == 'multiclass':
                    y_pred_val = model.predict(X_val)          # shape (n, n_classes)
                    test_pred += model.predict(X_test) / self.n_splits
            elif self.task_type == 'regression':
                y_pred_val = model.predict(X_val)
                test_pred += model.predict(X_test) / self.n_splits
            elif self.task_type == 'binary':
                y_pred_val = model.predict_proba(X_val)[:, 1]
                test_pred += model.predict_proba(X_test)[:, 1] / self.n_splits
            elif self.task_type == 'multiclass':
                y_pred_val = model.predict_proba(X_val)
                test_pred += model.predict_proba(X_test) / self.n_splits

            oof_pred[valid_id] = y_pred_val
            score = self.ScoreMetric(y_val, y_pred_val)
            print(f'  AUC: {score:.6f}')
            self.scores.loc[model_name, f'Fold {n_fold+1}'] = score

            # ✅ 每個 fold 結束後釋放記憶體
            gc.collect()

        self.scores.loc[model_name, 'Score'] = self.scores.loc[model_name, 'Fold 1':'Fold 5'].mean()
        return oof_pred, test_pred

    def _pred_cols(self, model_name):
        if self.task_type == 'multiclass':
            return [f'{model_name}_{i}' for i in range(self.n_classes)]
        return [model_name]

    def _save_preds(self, model_name, oof_pred, test_pred):
        cols = self._pred_cols(model_name)
        pd.DataFrame(oof_pred,  columns=cols).to_csv(f'{model_name}_oof.csv',  index=False)
        pd.DataFrame(test_pred, columns=cols).to_csv(f'{model_name}_test.csv', index=False)

    def _load_preds(self, model_name):
        oof  = pd.read_csv(f'{Config.DATA_DIR}/{model_name}_oof.csv')
        test = pd.read_csv(f'{Config.DATA_DIR}/{model_name}_test.csv')
        return oof.values, test.values

    def _append_preds(self, model_name, oof_pred, test_pred):
        cols = self._pred_cols(model_name)
        self.OOF_preds  = pd.concat([self.OOF_preds,  pd.DataFrame(oof_pred,  columns=cols)], axis=1).reset_index(drop=True)
        self.TEST_preds = pd.concat([self.TEST_preds, pd.DataFrame(test_pred, columns=cols)], axis=1).reset_index(drop=True)

    def run(self):
        for model_name, model in tqdm(self.models.items()):
            if self.training:
                oof_pred, test_pred = self.train(model, self.X.copy(), self.y, self.test.copy(), model_name)
                self._save_preds(model_name, oof_pred, test_pred)
            else:
                oof_pred, test_pred = self._load_preds(model_name)
                for n_fold, (train_id, valid_id) in enumerate(self.folds.split(oof_pred, self.y)):
                    y_pred_val, y_val = oof_pred[valid_id], self.y.iloc[valid_id]
                    self.scores.loc[model_name, f'Fold {n_fold+1}'] = self.ScoreMetric(y_val, y_pred_val)
                self.scores.loc[model_name, 'Score'] = self.scores.loc[model_name, 'Fold 1':'Fold 5'].mean()
            self._append_preds(model_name, oof_pred, test_pred)
            gc.collect()

        if len(self.models) > 1:
            if self.task_is_regression:
                meta_model = LinearRegression()
            elif self.task_type == 'multiclass':
                meta_model = LogisticRegression(multi_class='multinomial', class_weight='balanced',
                                                max_iter=2000, random_state=42, C=0.05)
            else:
                meta_model = LogisticRegression(max_iter=2000, random_state=42, C=0.1)

            OOF  = logit(np.clip(self.OOF_preds,  1e-15, 1-1e-15))
            TEST = logit(np.clip(self.TEST_preds, 1e-15, 1-1e-15))
            meta_model.fit(OOF, self.y)
            Ensemble_TEST = meta_model.predict_proba(TEST)[:, 1]
            Ensemble_OOF, _ = self.train(meta_model, OOF, self.y, TEST, 'Ensemble')
            self.scores.loc['Ensemble', 'Score'] = self.ScoreMetric(self.y, Ensemble_OOF)
            self.scores = self.scores.sort_values('Score')
            self.score_bar()
            self.plot_result(Ensemble_OOF)
            return Ensemble_TEST, Ensemble_OOF
        else:
            model_name = list(self.models.keys())[0]
            print(Style.BRIGHT+Fore.GREEN+f'{model_name} score {self.scores.loc[model_name, "Score"]:.7f}\n')
            self.plot_result(self.OOF_preds.iloc[:, 0].values)
            return self.TEST_preds.values, self.OOF_preds.values

    def score_bar(self):
        plt.figure(figsize=(16, max(6, len(self.scores)*0.4)))
        colors = ['#ef5350' if i == 'Ensemble' else '#3cb371' for i in self.scores.Score.index]
        hbars  = plt.barh(self.scores.index, self.scores.Score, color=colors, height=0.6)
        plt.bar_label(hbars, fmt='%.6f')
        plt.tight_layout()
        plt.show()

    def plot_result(self, oof):
        if self.n_classes == 2:
            y_bin = pd.DataFrame({self.labels[0]: (self.y==0).astype(int),
                                  self.labels[1]: (self.y==1).astype(int)})
            probs = np.vstack([1-oof, oof]).T
            y_pred= (probs[:, 1] >= 0.5).astype(int)
        else:
            y_bin = pd.DataFrame(label_binarize(self.y, classes=range(self.n_classes)), columns=self.labels)
            probs = oof
            y_pred= np.argmax(probs, axis=1)

        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        axes_flat = axes.ravel()
        colors = ['#3cb371','#ef5350','#5c9ded','#ffa726','#ab47bc']

        ax_roc = axes_flat[0]
        for i, name in enumerate(self.labels):
            RocCurveDisplay.from_predictions(y_bin.iloc[:,i], probs[:,i],
                                             name=name, ax=ax_roc, color=colors[i])
        ax_roc.plot([0,1],[0,1],'--k')
        ax_roc.set_title('ROC (one-vs-rest)')
        ax_roc.legend(loc='lower right')

        ConfusionMatrixDisplay.from_predictions(
            self.y, y_pred, display_labels=self.labels,
            colorbar=False, ax=axes_flat[1], cmap='Greens')
        axes_flat[1].set_title('Confusion Matrix')

        ax_pr = axes_flat[2]
        for i, name in enumerate(self.labels):
            precision, recall, _ = precision_recall_curve(y_bin.iloc[:,i], probs[:,i])
            ap = average_precision_score(y_bin.iloc[:,i], probs[:,i])
            ax_pr.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=colors[i])
        ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
        ax_pr.set_title('Precision-Recall Curves'); ax_pr.legend()

        ax_cc = axes_flat[3]
        bins = np.linspace(0, 1, 20)
        for cls, color in zip(range(self.n_classes), colors):
            p = probs[:,cls]
            is_cls = y_bin.iloc[:,cls].values
            bcs, bps = [], []
            for i in range(len(bins)-1):
                mask = (p >= bins[i]) & (p < bins[i+1])
                if mask.sum() > 0:
                    bcs.append((bins[i]+bins[i+1])/2)
                    bps.append(is_cls[mask].mean())
            ax_cc.plot(bcs, bps, 'o-', color=color, label=self.labels[cls])
        ax_cc.plot([0,1],[0,1],'--k')
        ax_cc.set_title('Calibration Curves'); ax_cc.legend()
        plt.tight_layout()
        plt.show()

In [32]:
models_filtered = {k: v for k, v in models.items() if k == 'LGBM3_2'}
trainer = Trainer(X, y, test, models_filtered, num_features, cat_features, training=True)
TEST_preds, OOF_preds = trainer.run()

  0%|          | 0/1 [00:00<?, ?it/s]

LGBM3_2


  0%|          | 0/1 [00:12<?, ?it/s]

Fold 1


AttributeError: type object 'LGBMClassifier' has no attribute 'train'

## Submission

In [ ]:
print(test.columns.tolist())
print(test.head())

['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'Driver_org_mean', 'Compound_org_mean', 'Race_org_mean', 'Year_org_mean', 'PitStop_org_mean', 'LapNumber_org_mean', 'Stint_org_mean', 'TyreLife_org_mean', 'Position_org_mean', 'LapTime (s)_org_mean', 'LapTime_Delta_org_mean', 'Cumulative_Degradation_org_mean', 'RaceProgress_org_mean', 'Position_Change_org_mean', 'Race_Year_comb', 'Race_Compound_comb', 'TyreLife_LapNumber_ratio', 'LapNumber_RaceProgress_ratio', 'LapTime (s)_*_Cumulative_Degradation', 'LapTime (s)_*_Cumulative_Degradation_abs', 'LapTime (s)_Cumulative_Degradation_abs_ratio', 'RaceProgress_200_quantile_bin', 'LapTime (s)_7_quantile_bin', 'Year_cat', 'PitStop_cat', 'LapNumber_cat', 'Stint_cat', 'TyreLife_cat', 'Position_cat', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'RaceProgress_cat', 'Position_Change_cat', 

In [ ]:
submission = pd.DataFrame({
    'id': test.index,
    'PitNextLap': TEST_preds.flatten()
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,PitNextLap
0,439140,0.007510
1,439141,0.021223
2,439142,0.005879
3,439143,0.153990
4,439144,0.778781


In [ ]:
submission = Config.submission.copy()
submission[Config.target] = TEST_preds
submission.to_csv('submission.csv', index=False)

def plot_cdf(values, label, color, ax):
    sv = np.sort(values)
    yv = np.arange(1, len(sv)+1) / len(sv)
    ax.plot(yv, sv, label=label, color=color)

display(submission.head())
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

plot_cdf(y,             'Target', 'blue',     axes[0])
plot_cdf(OOF_preds,     'Train',  '#3cb371',  axes[0])
plot_cdf(submission[Config.target], 'Test', 'red', axes[0])
axes[0].legend(); axes[0].set_title('CDF: Train / Test')

sns.kdeplot(submission[Config.target], label='Test', fill=True, color='red',     alpha=1.0, ax=axes[1])
sns.kdeplot(OOF_preds,                 label='Train',fill=True, color='#3cb371', alpha=0.5, ax=axes[1])
axes[1].set_title('Distribution of Predictions'); axes[1].legend()

plt.tight_layout()
plt.show()
print('✅ submission.csv 已儲存')